In [1]:
import sys
!{sys.executable} -m pip install opencv-python
!{sys.executable} -m pip install IMAGEIO
!{sys.executable} -m pip install albumentations
!{sys.executable} -m pip install -q clodsa
!{sys.executable} -m pip install -q git+https://github.com/aleju/imgaug.git
!{sys.executable} -m pip install imagecorruptions

In [45]:
!pip install future
!pip install clodsa


In [3]:
import os
import numpy as np
import cv2 
from glob import glob
from tqdm import tqdm
import imageio
import albumentations as A
from albumentations import *
import sklearn
from sklearn import *
import pandas as pd
from matplotlib import pyplot as plt
import random
import imageio
import imgaug as ia
import imgaug.augmenters as iaa

ModuleNotFoundError: No module named 'imgaug'

In [ ]:
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [ ]:
def load_data(path):
    # x = images and y = masks
    images_0 = sorted(glob(os.path.join(path, "images_0", "*.tif")))
    masks_0 = sorted(glob(os.path.join(path, "masks_0", "*.gif")))
    images_1 = sorted(glob(os.path.join(path, "images_1", "*.tif")))
    masks_1 = sorted(glob(os.path.join(path, "masks_1", "*.gif")))
    images_2 = sorted(glob(os.path.join(path, "images_2", "*.JPG")))
    masks_2 = sorted(glob(os.path.join(path, "masks_2", "*.tif")))
    images_3 = sorted(glob(os.path.join(path, "images_3", "*.jpg")))
    masks_3 = sorted(glob(os.path.join(path, "masks_3", "*.png")))
    images_4 = sorted(glob(os.path.join(path, "images_4", "*.ppm")))
    masks_4 = sorted(glob(os.path.join(path, "masks_4", "*.ppm")))


    return (images_0, masks_0), (images_1, masks_1), (images_2, masks_2), (images_3, masks_3), (images_4, masks_4)

In [ ]:
def process_data(images, mask, save_path, stage, ma_filetype_gif = False):
    IMG_H = 512
    IMG_W = 512
    index = 1
    if stage == 0: 
        index = 1
    elif stage == 1: 
        index = 21
    elif stage == 2:
        index = 41
    elif stage == 3: 
        index = 56
    else:
        index = 84
    for idx, (x, y) in tqdm(enumerate(zip(images, mask)), total= len(images)):
        
        #get image
        
        x = cv2.imread(x, cv2.IMREAD_GRAYSCALE)
   
        if ma_filetype_gif == True:
            y = imageio.mimread(y)[0]

        else:
            y = cv2.imread(y, cv2.IMREAD_GRAYSCALE)
            

        X = [x]
        Y = [y]

        
        for (i, m) in zip(X, Y):
           # i = cv2.resize(i, (IMG_W, IMG_H))
           # m = cv2.resize(m, (IMG_W, IMG_H))
            

            i = cv2.resize(i, (IMG_W, IMG_H))
            m = cv2.resize(m, (IMG_W, IMG_H))
            
            
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            i = clahe.apply(x)
            m = clahe.apply(y)

            
            tmp_image_name = f"{index}.jpg"
            tmp_mask_name = f"{index}.jpg"

            image_path = os.path.join(save_path, "image_total", tmp_image_name)
            mask_path = os.path.join(save_path, "mask_total", tmp_mask_name)

            cv2.imwrite(image_path, i)
            cv2.imwrite(mask_path, m)

            index = index + 1

In [5]:
(images_0, masks_0), (images_1, masks_1), (images_2, masks_2), (images_3, masks_3), (images_4, masks_4) = load_data('/Users/sadda/Documents/Science_Fair/Datasets/RSEG/')
create_dir('/Users/sadda/Documents/Science_Fair/Datasets/RSEG/image_total/')
create_dir('/Users/sadda/Documents/Science_Fair/Datasets/RSEG/mask_total/')

In [12]:
save_path_data = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/'
process_data(images_0, masks_0, save_path_data, stage = 0, ma_filetype_gif = True)
process_data(images_1, masks_1, save_path_data, stage = 1, ma_filetype_gif = True)
process_data(images_2, masks_2, save_path_data, stage = 2)
process_data(images_3, masks_3, save_path_data, stage = 3)
process_data(images_4, masks_4, save_path_data, stage = 4)


NameError: name 'images_0' is not defined

In [2]:
def load_data_tt(path):
    images = sorted(glob(os.path.join(path, "image_total", "*.jpg")))
    masks = sorted(glob(os.path.join(path, "mask_total", "*.jpg")))
    df = pd.DataFrame(images)
    df[1] = pd.DataFrame(masks)
    train_df, test_df = sklearn.model_selection.train_test_split(df, test_size=0.2, train_size=0.8, random_state=15, shuffle=True)
    train_im = list(train_df[0])
    train_ma = list(train_df[1])
    test_im = list(test_df[0])
    test_ma = list(test_df[1])
    return (train_im, train_ma), (test_im, test_ma)    

In [3]:
def train_test_data(images, mask, save_path):

    IMG_H = 512
    IMG_W = 512
    
    for idx, (x, y) in tqdm(enumerate(zip(images, mask)), total= len(images)):

        name = x.split('/')[-1].split('.')[0]

        #get image 
        x = cv2.imread(x, cv2.IMREAD_GRAYSCALE)
        y = cv2.imread(y, cv2.IMREAD_GRAYSCALE)
            


        X = [x]
        Y = [y]

        for i, m in zip(X, Y):
            i = cv2.resize(i, (IMG_W, IMG_H))
            m = cv2.resize(m, (IMG_W, IMG_H))

            tmp_image_name = f"{name}.jpg"
            tmp_mask_name = f"{name}.jpg"


            image_path = os.path.join(save_path, "image", tmp_image_name)
            mask_path = os.path.join(save_path, "mask", tmp_mask_name)

            cv2.imwrite(image_path, i)
            cv2.imwrite(mask_path, m)



In [71]:
(train_im, train_ma), (test_im, test_ma) = load_data_tt('/Users/sadda/Documents/Science_Fair/Datasets/RSEG')
# (train_im, train_ma), (test_im, test_ma)  
train_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/train/'
train_test_data(train_im, train_ma, train_path)
test_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/test/'
train_test_data(test_im, test_ma, test_path)

100%|██████████████████████████████████████████| 21/21 [00:00<00:00, 155.47it/s]


In [4]:
def augment_data(images, mask, save_path, augment=True):
    IMG_H = 512
    IMG_W = 512

    for idx, (x, y) in tqdm(enumerate(zip(images, mask)), total= len(images)):
        # name extraction
        name = x.split('/')[-1].split('.')[0]

        #get image
        x = cv2.imread(x, cv2.IMREAD_GRAYSCALE)
        y = cv2.imread(y, cv2.IMREAD_GRAYSCALE)



        if augment == True:
            aug = HorizontalFlip(p = 1.0)
            augmented = aug(image = x, mask = y)
            x1 = augmented['image']
            y1 = augmented['mask']

            aug = VerticalFlip(p=1.0)
            augmented = aug(image=x, mask=y)
            x2 = augmented['image']
            y2 = augmented['mask']

            aug = GaussianBlur(blur_limit=(3,7), sigma_limit=0, p=1)
            augmented = aug(image=x, mask=y)
            x3 = augmented['image']
            y3 = augmented['mask']

            aug = A.Compose([
            A.OneOf([
            A.GridDistortion(p=0.5),
            A.ElasticTransform(alpha=120, sigma=120 * 0.05, alpha_affine=120 * 0.03, p=0.5),
            A.OpticalDistortion(distort_limit=2, shift_limit=0.5, p=1)
            ], p=0.8)])
            augmented = aug(image=x, mask=y)
            x4 = augmented['image']
            y4 = augmented['mask']

            aug = A.Compose([
            A.OneOf([
            A.PixelDropout(p=0.02),
            A.RandomGamma(p=1)
            ], p=0.8)])
            augmented = aug(image=x, mask=y)
            x5 = augmented['image']
            y5 = augmented['mask']
            
            aug = RandomSizedCrop(min_max_height=(128, 448), height=IMG_H, width=IMG_W, p=1)
            augmented = aug(image = x, mask = y)
            x6 = augmented['image']
            y6 = augmented['mask']
            
            aug = Rotate(limit=75, interpolation=1, border_mode=4, rotate_method='largest_box', p=1)
            augmented = aug(image = x, mask = y)
            x7 = augmented['image']
            y7 = augmented['mask']
            
            aug = Transpose(p=1)
            augmented = aug(image = x, mask = y)
            x8= augmented['image']
            y8= augmented['mask']

            aug = A.Compose([
            A.OneOf([
            A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=0.5),
            A.RandomBrightnessContrast(p = 1.0)
            ], p=0.8)])
            augmented = aug(image=x, mask=y)
            x9 = augmented['image']
            y9 = augmented['mask']
            
            aug = JpegCompression(quality_lower=96, quality_upper=100, p=1)
            augmented = aug(image = x, mask = y)
            x10= augmented['image']
            y10= augmented['mask']

            
            
            

            X = [x, x1, x2, x3, x4, x5, x6, x7, x8, x9, x10]
            Y = [y, y1, y2, y3, y4, y5, y6, y7, y8, y9, y10]


        index = 0
        for i, m in zip(X, Y):
            i = cv2.resize(i, (IMG_W, IMG_H))
            m = cv2.resize(m, (IMG_W, IMG_H))


            tmp_image_name = f"{name}_{index}.jpg"
            tmp_mask_name = f"{name}_{index}.jpg"

            image_path = os.path.join(save_path, "image", tmp_image_name)
            mask_path = os.path.join(save_path, "mask", tmp_mask_name)

            cv2.imwrite(image_path, i)
            cv2.imwrite(mask_path, m)

            index = index + 1


In [78]:
path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/train/'
save_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/new_data/aug_train/'
create_dir(os.path.join(save_path, 'image'))
create_dir(os.path.join(save_path, 'mask'))

def load_data_aug(path):
    images = sorted(glob(os.path.join(path, "images", "*.jpg")))
    masks = sorted(glob(os.path.join(path, "labels", "*.jpg")))
    return (images, masks)
(images, masks) = load_data_aug(path)
augment_data(images, masks, save_path, augment=True)

100%|███████████████████████████████████████████| 82/82 [00:03<00:00, 22.91it/s]


In [4]:
## new imports

import skimage.io as io
import skimage.transform as trans
import tensorflow as tf
from tensorflow.keras.models import *
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.callbacks import ModelCheckpoint, LearningRateScheduler
from tensorflow.keras import backend as keras
import matplotlib.pyplot as plt
import scipy.misc as sc
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, ReduceLROnPlateau, EarlyStopping, TensorBoard
from tensorflow.keras.optimizers.legacy import Adam
from tensorflow.keras.metrics import Recall, Precision, Accuracy



In [2]:

def adjustData(img,mask,flag_multi_class,num_class):
    if(flag_multi_class):
        img = img/255
        mask = mask[:,:,:,0] if(len(mask.shape) == 4) else mask[:,:,0]
        new_mask = np.zeros(mask.shape + (num_class,))
        for i in range(num_class):
            #for one pixel in the image, find the class in mask and convert it into one-hot vector
            #index = np.where(mask == i)
            #index_mask = (index[0],index[1],index[2],np.zeros(len(index[0]),dtype = np.int64) + i) if (len(mask.shape) == 4) else (index[0],index[1],np.zeros(len(index[0]),dtype = np.int64) + i)
            #new_mask[index_mask] = 1
            new_mask[mask == i,i] = 1
        new_mask = np.reshape(new_mask,(new_mask.shape[0],new_mask.shape[1]*new_mask.shape[2],new_mask.shape[3])) if flag_multi_class else np.reshape(new_mask,(new_mask.shape[0]*new_mask.shape[1],new_mask.shape[2]))
        mask = new_mask
    elif(np.max(img) > 1):
        img = img / 255
        mask = mask /255
        mask[mask > 0.5] = 1
        mask[mask <= 0.5] = 0
        #print(np.shape(mask),np.shape(img))
    return (img,mask)



def trainGenerator(batch_size,train_path,image_folder,mask_folder,aug_dict,image_color_mode = "grayscale",
                    mask_color_mode = "grayscale",image_save_prefix  = "image",mask_save_prefix  = "mask",
                    flag_multi_class = False,num_class = 2,save_to_dir = None,target_size = (512,512),seed = 2):
    '''
    can generate image and mask at the same time
    use the same seed for image_datagen and mask_datagen to ensure the transformation for image and mask is the same
    if you want to visualize the results of generator, set save_to_dir = "your path"
    '''
    image_datagen = ImageDataGenerator(**aug_dict)
    mask_datagen = ImageDataGenerator(**aug_dict)
    image_generator = image_datagen.flow_from_directory(
        train_path,
        classes = [image_folder],
        class_mode = None,
        color_mode = image_color_mode,
        target_size = target_size,
        batch_size = batch_size,
        save_to_dir = save_to_dir,
        save_prefix  = image_save_prefix,
        seed = seed)
    mask_generator = mask_datagen.flow_from_directory(
        train_path,
        classes = [mask_folder],
        class_mode = None,
        color_mode = mask_color_mode,
        target_size = target_size,
        batch_size = batch_size,
        save_to_dir = save_to_dir,
        
        save_prefix  = mask_save_prefix,
        seed = seed)
    train_generator = zip(image_generator, mask_generator)
    for (img,mask) in train_generator:
        img,mask = adjustData(img,mask,flag_multi_class,num_class)
        yield (img,mask)


def testGenerator(test_path,num_image = 30,target_size = (512,512),flag_multi_class = False,as_gray = True):
    image_datagen = ImageDataGenerator(rescale=1./255)
    mask_datagen = ImageDataGenerator(rescale=1./255)
    files=sorted(os.listdir(test_path))
    num_image=len(files)
    for i in range(num_image):
        img = io.imread(os.path.join(test_path,files[i]),as_gray = as_gray)
        #img = cv2.imread(os.path.join(test_path,files[i]), cv2.IMREAD_GRAYSCALE)
        print(files[i])
        img = trans.resize(img,target_size)
        img = np.reshape(img,img.shape+(1,)) if (not flag_multi_class) else img
        img = np.reshape(img,(1,)+img.shape)
        yield img
        


def validGenerator(batch_size,test_path,image_folder,mask_folder,aug_dict,image_color_mode = "grayscale",
                    mask_color_mode = "grayscale",image_save_prefix  = "image",mask_save_prefix  = "mask",
                    flag_multi_class = False,num_class = 2,save_to_dir = None,target_size = (512,512),seed = 1):
    image_datagen = ImageDataGenerator(**aug_dict)
    mask_datagen = ImageDataGenerator(**aug_dict)
    image_generator = image_datagen.flow_from_directory(
      test_path,
      classes = [image_folder],
      class_mode = None,
      color_mode = image_color_mode,
      target_size = target_size,
      batch_size = batch_size,
      save_to_dir = save_to_dir,
      save_prefix  = image_save_prefix,
      seed = seed)
    mask_generator = mask_datagen.flow_from_directory(
      test_path,
      classes = [mask_folder],
      class_mode = None,
      color_mode = mask_color_mode,
      target_size = target_size,
      batch_size = batch_size,
      save_to_dir = save_to_dir,
      save_prefix  = mask_save_prefix,
      seed = seed)
    valid_generator = zip(image_generator, mask_generator)
    for(img,mask) in valid_generator:
        img,mask = adjustData(img,mask,flag_multi_class,num_class)
        yield (img,mask)

In [14]:
def labelVisualize(num_class,color_dict,img): # save test images
    img = img[:,:,0] if len(img.shape) == 3 else img
    img_out = np.zeros(img.shape + (3,))
    for i in range(num_class):
        img_out[img == i] = color_dict[i]
      
    return img_out


def saveResult(img_path,save_path,npyfile,flag_multi_class = False,num_class = 2):
    files=sorted(os.listdir(img_path))
    #print(len(img_path))
    #print(len(npyfile))
    
    for i,item in enumerate(npyfile):
        img = labelVisualize(num_class,COLOR_DICT,item) if flag_multi_class else item[:,:,0]
        #img1=np.array(((img - np.min(img))/np.ptp(img))>0.6).astype(float)
        img[img>0.1]=1
        img[img<=0.1]=0
        io.imsave(os.path.join(save_path, files[i]+'_predict.png'),img)
        


SyntaxError: invalid syntax (1638150315.py, line 16)

In [6]:
#metrics
from sklearn.metrics import accuracy_score, f1_score, jaccard_score, precision_score, recall_score

def mean_iou(true_y, pred_y):#iou
    def f(true_y, pred_y):
        intersection = (true_y*pred_y).sum()
        union = true_y.sum() + pred_y.sum() - intersection
        x = (intersection + 1e-15) / (union + 1e-15)
        x = x.astype(np.float32)
        return x
    return tf.numpy_function(f, [true_y, pred_y], tf.float32)

smooth = 1e-15
def dice_coef(true_y, pred_y):
    true_y = tf.keras.layers.Flatten()(true_y)
    pred_y = tf.keras.layers.Flatten()(pred_y)
    intersection = tf.reduce_sum(true_y * pred_y)
    return (2. * intersection + smooth) / (tf.reduce_sum(true_y) + tf.reduce_sum(pred_y) + smooth)

def dice_loss(true_y, pred_y):
    return 1.0 - dice_coef(true_y, pred_y)


def dice(true_mask, pred_mask, non_seg_score=1.0):
    assert true_mask.shape == pred_mask.shape

    true_mask = np.asarray(true_mask).astype(np.bool)
    pred_mask = np.asarray(pred_mask).astype(np.bool)

    # If both segmentations are all zero, the dice will be 1. (Developer decision)
    #im_sum = true_mask.sum() + pred_mask.sum()
    #if im_sum == 0:
    #    return non_seg_score

    # Compute Dice coefficient
    intersection = np.logical_and(true_mask, pred_mask)
    return 2. * intersection.sum() / im_sum
     

def mean_dice(true_path, pred_path):
  
  sum = 0
  
  for img in sorted(os.listdir(pred_path)):
    
    true_tmp = cv2.imread(true_path + '/' + img, 0)
    pred_tmp = cv2.imread(pred_path + '/' + img, 0)
    
    a = dice(true_tmp, pred_tmp)
    print(a)
    sum += a
  
  return sum/len(os.listdir(true_path))

In [9]:
data_gen_args = dict() 

In [10]:
pip install keras-unet-collection==0.1.13

Note: you may need to restart the kernel to use updated packages.


In [11]:
from keras_unet_collection import models, losses

import skimage.io as io
import skimage.transform as trans
import tensorflow as tf
from tensorflow.keras.models import *
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.callbacks import ModelCheckpoint, LearningRateScheduler
from tensorflow.keras import backend as keras
import matplotlib.pyplot as plt
import scipy.misc as sc
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, ReduceLROnPlateau, EarlyStopping, TensorBoard
from tensorflow.keras.optimizers.legacy import Adam
from tensorflow.keras.metrics import Recall, Precision, Accuracy



model = models.att_unet_2d((512, 512, 1), filter_num=[64, 128, 256, 512, 1024], n_labels=1, 
                           stack_num_down=2, stack_num_up=2, activation='ReLU', 
                           atten_activation='ReLU', attention='add', output_activation='Sigmoid', 
                           batch_norm=True, pool=True, unpool=False, 
                           name='attunet')


model.compile(loss='binary_crossentropy', optimizer=Adam(lr = 1e-4), 
              metrics=['accuracy', Precision(), dice_coef])

print(model.summary())


Metal device set to: Apple M1


2023-02-12 11:36:02.834093: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2023-02-12 11:36:02.834286: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "attunet_model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 512, 512, 1  0           []                               
                                )]                                                                
                                                                                                  
 attunet_down0_0 (Conv2D)       (None, 512, 512, 64  576         ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 attunet_down0_0_bn (BatchNorma  (None, 512, 512, 64  256        ['attunet_down0_0[0][0]']        
 lization)                      )                                                     

                                                                                                  
 attunet_down3_conv_1_activatio  (None, 64, 64, 512)  0          ['attunet_down3_conv_1_bn[0][0]']
 n (ReLU)                                                                                         
                                                                                                  
 attunet_down4_encode_maxpool (  (None, 32, 32, 512)  0          ['attunet_down3_conv_1_activation
 MaxPooling2D)                                                   [0][0]']                         
                                                                                                  
 attunet_down4_conv_0 (Conv2D)  (None, 32, 32, 1024  4718592     ['attunet_down4_encode_maxpool[0]
                                )                                [0]']                            
                                                                                                  
 attunet_d

                                                                                                  
 attunet_up1_decode_activation   (None, 128, 128, 25  0          ['attunet_up1_decode_bn[0][0]']  
 (ReLU)                         6)                                                                
                                                                                                  
 attunet_up1_att_theta_x (Conv2  (None, 128, 128, 12  32896      ['attunet_down2_conv_1_activation
 D)                             8)                               [0][0]']                         
                                                                                                  
 attunet_up1_att_phi_g (Conv2D)  (None, 128, 128, 12  32896      ['attunet_up1_decode_activation[0
                                8)                               ][0]']                           
                                                                                                  
 attunet_u

 attunet_up2_conv_after_concat_  (None, 256, 256, 12  294912     ['attunet_up2_concat[0][0]']     
 0 (Conv2D)                     8)                                                                
                                                                                                  
 attunet_up2_conv_after_concat_  (None, 256, 256, 12  512        ['attunet_up2_conv_after_concat_0
 0_bn (BatchNormalization)      8)                               [0][0]']                         
                                                                                                  
 attunet_up2_conv_after_concat_  (None, 256, 256, 12  0          ['attunet_up2_conv_after_concat_0
 0_activation (ReLU)            8)                               _bn[0][0]']                      
                                                                                                  
 attunet_up2_conv_after_concat_  (None, 256, 256, 12  147456     ['attunet_up2_conv_after_concat_0
 1 (Conv2D

/Users/sadda/miniconda3/envs/tensorflow/lib/python3.10/site-packages/keras/optimizers/optimizer_v2/adam.py:117: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


In [8]:
train_batch = 3
valid_batch = 3
train_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/new_data/aug_train'
valid_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/new_data/test'
#init_model_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/model/pretrained'
#save_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/model/new_VSEGmodel'
csv_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/model/CSV/data2.csv'
num_train = 902
num_valid = 21
img_train_fpath = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/new_data/aug_train/final_img/'
img_valid_fpath = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/new_data/test/final_img'


train_generator = trainGenerator(train_batch, train_path, 'image', 'mask', data_gen_args, save_to_dir = None, target_size=(512,512))
valid_generator = validGenerator(valid_batch, valid_path, 'image', 'mask', data_gen_args, save_to_dir = None, target_size=(512,512))

#input_shape = (512,512,1)
#model = Attention_ResUNet(input_shape, NUM_CLASSES=1, dropout_rate=0.2, batch_norm=True)
#  if initial_model_path != None:
model.load_weights('AttResUNET2.hdf5')

#model = residual_attentionunet(input_shape)
#  if initial_model_path != None:
#    model.load_weights(initial_model_path)



In [12]:
callbacks = [
    #ModelCheckpoint(save_path, verbose=1, save_best_only=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.1, patience=5, min_lr=1e-6, verbose=1),
    CSVLogger(csv_path),
    TensorBoard(),
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=False)
    ]

model_checkpoint = tf.keras.callbacks.ModelCheckpoint('AttNet.hdf5', monitor='loss',verbose=1, save_best_only=True)

model.fit(train_generator,steps_per_epoch=num_train//train_batch,epochs=80,verbose=1,callbacks=[callbacks,model_checkpoint],
          validation_data=valid_generator, validation_steps=num_valid//valid_batch)

StopIteration: 

In [53]:
import sklearn.metrics as sm

def get_confusion_matrix_elements(groundtruth_list, predicted_list):
    """returns confusion matrix elements i.e TN, FP, FN, TP as floats
	See example code for helper function definitions
    """
    tn, fp, fn, tp = sm.confusion_matrix(groundtruth_list, predicted_list,labels=[0,1]).ravel()
    tn, fp, fn, tp = np.float64(tn), np.float64(fp), np.float64(fn), np.float64(tp)

    return tn, fp, fn, tp

def get_prec_rec_IoU_accuracy(groundtruth_list, predicted_list):
    """returns precision, recall, IoU and accuracy metrics
	"""
    tn, fp, fn, tp = get_confusion_matrix_elements(groundtruth_list, predicted_list)
    
    total = tp + fp + fn + tn
    accuracy = (tp + tn) / total
    prec=tp/(tp+fp)
    rec=tp/(tp+fn)
    IoU=tp/(tp+fp+fn)
    
    return prec,rec,IoU,accuracy

def get_f1_score(groundtruth_list, predicted_list):
    """Return f1 score covering edge cases"""

    tn, fp, fn, tp = get_confusion_matrix_elements(groundtruth_list, predicted_list)
    
    f1_score = (2 * tp) / ((2 * tp) + fp + fn)

    return f1_score

def get_validation_metrics(groundtruth,predicted):
    """Return all output metrics. Input is binary images"""
   
    u,v=np.shape(groundtruth)
    groundtruth_list=np.reshape(groundtruth,(u*v,))
    predicted_list=np.reshape(predicted,(u*v,))
    prec,rec,IoU,acc=get_prec_rec_IoU_accuracy(groundtruth_list, predicted_list)
    f1_score=get_f1_score(groundtruth_list, predicted_list)
   # print("Precision=",prec, "Recall=",rec, "IoU=",IoU, "acc=",acc, "F1=",f1_score)
    return prec,rec,IoU,acc,f1_score

def evalResult(gth_path,npyfile,target_size=(512,512),flag_multi_class = False,num_class = 2):
    files=sorted(os.listdir(gth_path))
    print(files)
    prec=0
    rec=0
    acc=0
    IoU=0
    f1_score=0
    for i,item in enumerate(npyfile):
        img = item[:,:,0]
        gth = io.imread(os.path.join(gth_path,files[i]))
        gth = trans.resize(gth,target_size)
        img1=np.array(((img - np.min(img))/np.ptp(img))>0.1).astype(float)
        gth1=np.array(((gth - np.min(gth))/np.ptp(gth))>0.1).astype(float)
        p,r,I,a,f=get_validation_metrics(gth1,img1)
        prec=prec+p
        rec=rec+r
        acc=acc+a
        IoU=IoU+I
        f1_score=f1_score+f
    print("Precision=",prec/(i+1), "Recall=",rec/(i+1), "IoU=",IoU/(i+1), "acc=",acc/(i+1), "F1=",f1_score/(i+1))    

In [ ]:
from tensorflow.keras import models, layers, regularizers
from tensorflow.keras import backend as K



In [81]:
input_shape = (512,512,1)
#model = Attention_ResUNet(input_shape, NUM_CLASSES=1, dropout_rate=0.2, batch_norm=True)
#  if initial_model_path != None:
#    model.load_weights(initial_model_path)

model2 = residual_attentionunet(input_shape)
model2= keras.models.load_model('retina_AttentionRESUnet_150epochs.hdf5')

NameError: name 'layers' is not defined

In [7]:
from tensorflow.keras.utils import CustomObjectScope
bv_model_path = '/Users/sadda/Documents/Science_Fair/Code/RSEG/final_model/AttNet.hdf5'
with CustomObjectScope({'dice_coef': dice_coef}):
    model_bv = tf.keras.models.load_model(bv_model_path)  

Metal device set to: Apple M1


2023-02-18 16:57:47.234067: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2023-02-18 16:57:47.234092: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [11]:
test_img_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/new_data/test/image/'
n_i=len(os.listdir(test_img_path))
test_gen = testGenerator(test_img_path)
results = model_bv.predict_generator(test_gen,n_i,verbose=1)
save_result = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/new_data/test/predicted_labels_2/'
saveResult(test_img_path,save_result,results)

13.jpg
14.jpg
 1/21 [>.............................] - ETA: 1s

/var/folders/jm/m054n7_s0bn2215whpnw82r06kh0mk/T/ipykernel_31185/2970205398.py:4: UserWarning: `Model.predict_generator` is deprecated and will be removed in a future version. Please use `Model.predict`, which supports generators.
  results = model_bv.predict_generator(test_gen,n_i,verbose=1)


16.jpg
21/21 [==============================] - 8s 376ms/step


Lossy conversion from float32 to uint8. Range [0, 1]. Convert image to uint8 prior to saving to suppress this warning.
Lossy conversion from float32 to uint8. Range [0, 1]. Convert image to uint8 prior to saving to suppress this warning.
Lossy conversion from float32 to uint8. Range [0, 1]. Convert image to uint8 prior to saving to suppress this warning.
Lossy conversion from float32 to uint8. Range [0, 1]. Convert image to uint8 prior to saving to suppress this warning.
Lossy conversion from float32 to uint8. Range [0, 1]. Convert image to uint8 prior to saving to suppress this warning.
Lossy conversion from float32 to uint8. Range [0, 1]. Convert image to uint8 prior to saving to suppress this warning.
Lossy conversion from float32 to uint8. Range [0, 1]. Convert image to uint8 prior to saving to suppress this warning.
Lossy conversion from float32 to uint8. Range [0, 1]. Convert image to uint8 prior to saving to suppress this warning.
Lossy conversion from float32 to uint8. Range [0

In [66]:
gt_path='/Users/sadda/Documents/Science_Fair/Datasets/RSEG/new_data/test/mask/'
evalResult(gt_path,results)

['13.jpg', '14.jpg', '16.jpg', '19.jpg', '25.jpg', '29.jpg', '33.jpg', '39.jpg', '40.jpg', '48.jpg', '55.jpg', '6.jpg', '61.jpg', '64.jpg', '67.jpg', '73.jpg', '82.jpg', '85.jpg', '90.jpg', '94.jpg', '97.jpg']
Precision= 0.8055706492920414 Recall= 0.8197802820296657 IoU= 0.6800337325961018 acc= 0.9619097028459821 F1= 0.8083484067052117
